We're having problems with noncompliance calculations that are leading me to think that maybe Ecology's model output files are mislabeled for different years. A really easy way to check this is to rely on the fact that all runs from the same years use the same hydrodynamic run. So the temperatures and salinities of any two runs for the same year should be identical.

Let's check our 2025 existing and reference runs to see if they agree on these.

In [1]:
case_file = '/gscratch/ssmc/USRS/PSI/Ben/projects/Ecy25_analysis/SSM_config_ecy25.yaml'

import sys
import warnings
from pathlib import Path

import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

sys.path.append('../../py_scripts')
import ssm_utils

In [2]:
ssm, case = ssm_utils.read_case(case_file)

Inferring case name to be ecy25


In [3]:
model_exist = ssm['paths']['model_output'][case][0]
model_ref = ssm['paths']['model_output'][case][1]
model_opt2_8 = ssm['paths']['model_output'][case][2]
ds_exist = xr.open_dataset(model_exist)
ds_ref = xr.open_dataset(model_ref)
ds_opt2_8 = xr.open_dataset(model_opt2_8)
ds_exist

<xarray.Dataset> Size: 292GB
Dimensions:  (Time: 8760, IJK: 160120)
Dimensions without coordinates: Time, IJK
Data variables: (12/52)
    Var_1    (Time, IJK) float32 6GB ...
    Var_2    (Time, IJK) float32 6GB ...
    Var_3    (Time, IJK) float32 6GB ...
    Var_4    (Time, IJK) float32 6GB ...
    Var_5    (Time, IJK) float32 6GB ...
    Var_6    (Time, IJK) float32 6GB ...
    ...       ...
    Var_47   (Time, IJK) float32 6GB ...
    Var_48   (Time, IJK) float32 6GB ...
    Var_49   (Time, IJK) float32 6GB ...
    Var_50   (Time, IJK) float32 6GB ...
    Var_51   (Time, IJK) float32 6GB ...
    Var_52   (Time, IJK) float32 6GB ...

Temperature is Var_18. Salinity is Var_19.

In [4]:
ds_exist['Var_18'].sel(Time=1234, IJK=123456) - ds_ref['Var_18'].sel(Time=1234, IJK=123456)

<xarray.DataArray 'Var_18' ()> Size: 4B
array(-0.00139999, dtype=float32)
Attributes:
    FVCOM_Name:  Temperature C

In [5]:
ds_exist['Var_19'].sel(Time=1234, IJK=123456) - ds_ref['Var_19'].sel(Time=1234, IJK=123456)

<xarray.DataArray 'Var_19' ()> Size: 4B
array(0.00110054, dtype=float32)
Attributes:
    FVCOM_Name:  Conc. of Salinity ppt

They don't match!!! Now before we jump to conclusions, let's use Opt2_8 as a positive control to ensure that temp/salinity should indeed be an exact match between runs.

In [6]:
ds_opt2_8['Var_18'].sel(Time=1234, IJK=123456) - ds_ref['Var_18'].sel(Time=1234, IJK=123456)

<xarray.DataArray 'Var_18' ()> Size: 4B
array(0., dtype=float32)
Attributes:
    FVCOM_Name:  Temperature C

In [7]:
ds_opt2_8['Var_19'].sel(Time=1234, IJK=123456) - ds_ref['Var_19'].sel(Time=1234, IJK=123456)

<xarray.DataArray 'Var_19' ()> Size: 4B
array(0., dtype=float32)
Attributes:
    FVCOM_Name:  Conc. of Salinity ppt

In [8]:
N = 1000

temps = np.zeros((N, 3), dtype=float)
salts = np.zeros((N, 3), dtype=float)

for i in range(N):
    t = np.random.randint(0, ds_exist.sizes['Time'])
    ijk = np.random.randint(0, ds_exist.sizes['IJK'])
    for j,m in enumerate((ds_exist, ds_ref, ds_opt2_8)):
        temps[i,j] = m['Var_18'].sel(Time=t, IJK=ijk).item()
        salts[i,j] = m['Var_19'].sel(Time=t, IJK=ijk).item()

display(temps[:,0:2].std(axis=1))
display(temps[:,1:3].std(axis=1))

array([8.36515427e-03, 1.24998093e-02, 1.15003586e-02, 7.40051270e-04,
       8.99982452e-03, 2.92501450e-02, 2.79998779e-02, 3.49998474e-04,
       4.40025330e-03, 2.65026093e-03, 6.49929047e-04, 1.01518631e-03,
       2.52723694e-05, 2.99978256e-03, 1.35940075e-01, 4.00018692e-03,
       2.50339508e-04, 5.50031662e-03, 1.64985657e-04, 1.24931335e-04,
       2.99930573e-04, 2.49481201e-03, 1.49726868e-04, 4.50134277e-04,
       4.45365906e-04, 4.70018387e-03, 2.49862671e-04, 4.30011749e-03,
       3.25498581e-02, 8.20159912e-04, 2.99930573e-04, 1.49726868e-04,
       2.44855881e-04, 9.39989090e-03, 1.56998634e-02, 1.00002289e-02,
       1.49965286e-03, 1.24497414e-02, 6.99996948e-04, 8.54969025e-03,
       6.48498535e-05, 8.24928284e-04, 1.80006027e-03, 1.25749111e-02,
       1.10483170e-03, 6.99996948e-04, 1.99794769e-04, 6.49929047e-04,
       5.00679016e-06, 4.01997566e-02, 9.51449871e-02, 9.84954834e-03,
       3.09991837e-03, 1.01995468e-03, 2.09999084e-03, 6.00004196e-03,
      

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [9]:
pd.DataFrame(temps, columns=('exist','reference','opt2_8')).to_excel('random_temps.xlsx')
pd.DataFrame(salts, columns=('exist','reference','opt2_8')).to_excel('random_salts.xlsx')